# 08 · Numba CUDA ①: Copy & 메모리 접근(coalescing)

> **CuPy 2일 집중 코스 — Day 2 / 단원 6 (Numba CUDA 커널 작성, GTC 06 기반)**

이제 **Numba CUDA**(`@cuda.jit`)로 커널을 **직접** 작성합니다. 가장 단순한 *복사 커널*로
07에서 배운 **스레드 인덱싱·coalescing·occupancy** 개념을 손으로 구현하고 성능 차이를 측정합니다.

### 왜 하필 "복사"로 시작하는가

`dst[i] = src[i]`는 계산이 전혀 없는, **순수하게 메모리 이동만** 하는 가장 단순한 커널입니다.
연산이 없으므로 실행 시간을 좌우하는 변수는 오직 **메모리 접근 패턴** 하나뿐입니다 — 즉 이
커널의 실행 시간은 곧 "이 GPU가 이 접근 패턴으로 낼 수 있는 유효 대역폭"을 그대로 드러내는
가장 깨끗한 측정 도구입니다. 07(5절)에서 개념으로만 설명했던 coalescing이 실제로 얼마나 큰
차이를 만드는지, 계산 로직이 섞이지 않은 이 예제만큼 명확하게 보여줄 예제는 없습니다. 이후
09(히스토그램)·11(RawKernel)의 더 복잡한 커널도 결국 "계산 + 메모리 접근"의 조합이므로, 여기서
메모리 접근만 따로 떼어 감을 잡아두면 이후 병목을 훨씬 빨리 알아챌 수 있습니다.

## 이 노트북에서 구현하는 개념 (07 참조)
- **2. 스레드 인덱싱**: `cuda.grid`, `blockDim/blockIdx/threadIdx`
- **5. coalescing**: blocked(흩어진) vs coalesced(연속) 접근 → 성능 비교
- **7. occupancy**: `threads_per_block`·`items_per_thread` 파라미터 스윕

## 학습 목표
- `@cuda.jit`로 커널을 정의하고 `kernel[blocks, threads](...)` 로 런치한다.
- 같은 복사 작업을 **메모리 접근 패턴**만 바꿔 가속한다.
- (선택) Nsight Compute로 메모리 처리량을 확인한다.

### 이 노트북의 흐름

Numba CUDA 기초(1절) → **blocked 복사 구현·측정**(2절, 기준선) → **coalesced 복사를 직접
작성**(3절, grid-stride 루프) → 두 접근의 성능 비교(4절) → `threads_per_block`을 바꿔가며
occupancy 영향 관찰(5절) → (선택) Nsight Compute로 실측 대역폭 확인(6절). 07의 "개념 → 노트북
매핑표"(8절)에서 08의 역할은 정확히 **coalescing을 손으로 구현하고 측정**하는 것입니다.


## 목차
1. [Numba CUDA 기초](#1)
2. [Blocked Copy (기준)](#2)
3. [Coalesced Copy (최적화)](#3)
4. [성능 비교](#4)
5. [파라미터 스윕(occupancy)](#5)
6. [(선택) Nsight Compute 프로파일](#6)
7. [체크포인트](#7)

> 필요 패키지: `numba`(CUDA 지원). `ncu`(Nsight Compute)는 6절에서만, 없으면 건너뜀.

In [ ]:
import numpy as np, cupy as cp, math
from numba import cuda
from course_utils import print_env, bench, gpu_ms, print_bench, compare
print_env()
print('Numba CUDA 사용 가능:', cuda.is_available())

<a id="1"></a>
## 1. Numba CUDA 기초

`@cuda.jit`로 장식한 Python 함수가 GPU 커널이 됩니다. 각 스레드는 `cuda.grid(1)`로 전역 인덱스를 얻습니다
(= `blockIdx.x*blockDim.x + threadIdx.x`, 07의 2절). 런치는 `kernel[그리드, 블록](인자)`.

### 조금 더 구체적으로

`@cuda.jit`은 데코레이트된 함수를 **처음 호출되는 순간** 해당 인자들의 타입에 맞춰 CUDA
바이너리로 즉석 컴파일(JIT)합니다 — Day 1(00 6.1절)에서 본 CuPy의 커널 캐싱과 정확히 같은
메커니즘입니다. 다만 차이가 하나 있습니다. CuPy(07의 A절)에서는 **커널 코드 자체를 CuPy가
생성**하고 사용자는 원소별 연산식만 넘겼다면, 여기서는 **스레드가 무엇을 할지(인덱싱·경계
검사·메모리 접근)를 전부 사용자가 직접** 씁니다. 그 대신 얻는 것은 완전한 제어권입니다.

- **`cuda.grid(1)`**: 1차원 grid에서의 전역 스레드 번호. 내부적으로
  `cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x`를 계산해줍니다(2D/3D는 `cuda.grid(2)`/`cuda.grid(3)`).
- **경계 검사(`if i < x.size`)가 필수인 이유**: 런치 구성은 보통
  `blocks = ceil(n / threads)`로 올림 계산하므로, 총 스레드 수(`blocks*threads`)가 배열 크기
  `n`보다 **커지는 경우가 일반적**입니다(나누어떨어지지 않으면 항상 그렇습니다). 경계를 검사하지
  않으면 남는 스레드가 배열 밖 메모리를 읽거나 쓰며 정의되지 않은 동작(out-of-bounds access)을
  일으킵니다 — 00(3.1절)에서 본 CuPy의 "인덱싱 wrap-around"와 달리, Numba CUDA 커널의 경계 밖
  접근은 조용히 넘어가지 않고 크래시나 메모리 손상으로 이어질 수 있어 더 위험합니다.
- **커널 코드가 쓸 수 있는 파이썬은 일부(subset)뿐**입니다 — 00(2절 하단)에서 미리 언급했던 그
  제약이 바로 여기서 실체를 드러냅니다. 리스트/딕셔너리 생성, 임의 객체·예외 처리, 대부분의
  파이썬 표준 라이브러리 호출은 커널 안에서 쓸 수 없고, 스칼라 수치 연산·배열 인덱싱·`math` 모듈의
  일부 함수 정도로 제한됩니다. 대신 그 대가로 host 쪽 파이썬 코드(`n`, `threads`, `blocks` 계산 등)는
  평소처럼 자유롭게 작성할 수 있습니다.
- **런치 문법 `kernel[blocks, threads](args)`**: 대괄호 안 두 값이 실행 구성(execution
  configuration)이고, 소괄호 안이 실제 커널 인자입니다. CuPy 배열을 그대로 넘기면 Numba가
  CUDA Array Interface를 통해 device 포인터를 인식합니다(추가 변환이나 복사 불필요).

> 💡 CuPy의 고수준 커널(07)과 Numba `@cuda.jit`(08~09)은 "추상화 수준"의 스펙트럼 양 끝에 있는
> 것이 아니라, **필요한 제어 수준에 따라 선택하는 도구**입니다. 단순 원소별/리덕션 연산이면
> `ElementwiseKernel`/`ReductionKernel`이 훨씬 짧고 안전하고, 인덱싱·공유메모리·atomic처럼
> 스레드 단위의 세밀한 제어가 필요해지면 Numba CUDA(또는 11의 CUDA C RawKernel)로 넘어옵니다.


In [ ]:
@cuda.jit
def scale_kernel(x, y, a):
    i = cuda.grid(1)              # 전역 인덱스 (2절)
    if i < x.size:               # 경계 검사
        y[i] = a * x[i]

n = 1 << 20
x = cp.arange(n, dtype=cp.float32); y = cp.empty_like(x)
threads = 256; blocks = (n + threads - 1) // threads
scale_kernel[blocks, threads](x, y, np.float32(2.0))
cp.cuda.Device().synchronize()
cp.testing.assert_allclose(cp.asnumpy(y), 2.0*cp.asnumpy(x)); print('OK')

<a id="2"></a>
## 2. Blocked Copy (기준)

각 스레드가 **연속된 `ipt`개**를 복사합니다. 한 스레드 입장에선 연속이지만, **같은 시점에 warp의 인접 스레드들은
멀리 떨어진 주소**를 읽습니다 → **흩어진(non-coalesced) 접근** (07의 5절). 메모리 효율이 낮습니다.

### 조금 더 구체적으로: 왜 "한 스레드는 연속"인데 "warp는 흩어짐"인가

아래 코드에서 스레드의 전역 인덱스를 `g = cuda.grid(1)`라 하면, 이 스레드는
`base = g*ipt`부터 `base+ipt-1`까지를 **자신만의 연속 구간**으로 처리합니다(`for k in
range(ipt)`). 문제는 **warp 안의 32개 스레드가 같은 시각에 어떤 주소를 건드리는가**입니다.
`threads=256, ipt=8`인 예제 셀 기준으로, 루프의 `k`번째 반복에서 warp(스레드 `g=0..31`)가
접근하는 주소는 다음과 같습니다.

| 스레드 | 담당 구간(base) | k=0 접근 주소 | k=1 접근 주소 |
|--------|------------------|----------------|----------------|
| g=0 | 0..7  | 0  | 1  |
| g=1 | 8..15 | 8  | 9  |
| g=2 | 16..23| 16 | 17 |
| ... | ...   | ...| ...|
| g=31| 248..255 | 248 | 249 |

즉 **같은 순간(같은 k)** 에 warp의 32개 스레드가 접근하는 주소는 `0, 8, 16, ..., 248`처럼
**8개(=`ipt`) float32 간격, 32바이트 간격**으로 흩어져 있습니다. GPU의 전역 메모리 컨트롤러는
접근을 32/64/128바이트 단위의 **트랜잭션(memory transaction)** 으로 처리하는데, 이 warp의
요청은 `0`부터 `248+3`(=251)까지 총 1KB 넘는 범위에 걸쳐 있어 **하나의 128바이트 트랜잭션에
담기지 않고 여러 번의 트랜잭션으로 쪼개집니다**. 결과적으로 필요한 유효 데이터량 대비 실제로
발생하는 메모리 트래픽이 몇 배로 늘어나고(트랜잭션 효율 저하), 이는 곧 실측 대역폭 저하로
이어집니다 — 4절에서 이 손실을 직접 숫자로 확인합니다.

> **비유**: 32명이 강당의 사물함(메모리)에서 물건을 하나씩 꺼내야 합니다. Blocked 방식은
> "1번 사람은 1~8번 칸, 2번 사람은 9~16번 칸..."처럼 **구역을 나눠 배정**하는 것과 같습니다.
> 같은 순간에는 각자 자기 구역의 첫 번째 물건만 꺼내므로, 32명이 강당 전체에 띄엄띄엄 흩어져
> 각자 따로 이동해야 합니다. 3절의 coalesced 방식은 반대로 "32명이 나란히 서서 한 줄(연속
> 32칸)을 한 번에 꺼내고, 다음 줄로 다 같이 이동"하는 방식입니다 — 이게 훨씬 효율적인 이유는
> 직관적으로도 명확합니다.


In [ ]:
@cuda.jit
def copy_blocked(src, dst, ipt):
    base = cuda.grid(1) * ipt
    for k in range(ipt):
        if base + k < src.size:
            dst[base + k] = src[base + k]

N = 1 << 24
src = cp.arange(N, dtype=cp.float32); dst = cp.empty_like(src)
threads = 256; ipt = 8
blocks_b = (N + threads*ipt - 1) // (threads*ipt)
copy_blocked[blocks_b, threads](src, dst, ipt)
cp.cuda.Device().synchronize()
cp.testing.assert_array_equal(src, dst); print('blocked OK')

<a id="3"></a>
## 3. Coalesced Copy (최적화) — 연습

**grid-stride 루프**로 바꿉니다: warp의 인접 스레드가 **인접 주소**를 읽도록(연속 접근), 그다음 `gridsize(1)`만큼
건너뛰며 반복. 이러면 메모리 트랜잭션이 합쳐져(coalesced) 대역폭을 제대로 씁니다.

### 조금 더 구체적으로: grid-stride 루프란

```
i = cuda.grid(1); stride = cuda.gridsize(1)
while i < n: ...; i += stride
```

이 패턴이 하는 일은 두 가지입니다.

1. **각 반복의 접근을 연속으로 만든다**: 특정 반복 시점에 스레드 `g`가 접근하는 주소는 항상
   `g + (반복 횟수)*stride`입니다. 같은 반복(같은 시점)에서 warp의 스레드 `g=0..31`이 접근하는
   주소는 `0,1,2,...,31` — **정확히 연속된 32개**입니다. 2절의 blocked 방식과 정반대로, 이번엔
   *warp 입장에서* 연속이고 *스레드 개인 입장에서*는 흩어져 있습니다(다음 원소가 `stride`만큼
   떨어져 있으므로). 하지만 성능을 결정하는 것은 **"같은 순간 warp가 어디를 보는가"**이지 "한
   스레드가 시간에 따라 어디를 도는가"가 아니므로, 이 방식이 훨씬 빠릅니다.
2. **런치 구성과 데이터 크기를 분리한다**: `stride = cuda.gridsize(1)`(=`blocks*threads`)만큼씩
   건너뛰므로, 총 스레드 수가 데이터 크기보다 작아도(`blocks_c=1024`처럼 임의로 정해도) `while`
   루프가 알아서 남은 부분까지 처리합니다. 그래서 데이터 크기 `N`이 바뀌어도 `blocks_c`를 다시
   계산할 필요가 없습니다 — 이는 CUDA 커널을 작성할 때 아주 흔히 쓰이는 관용구(idiom)로,
   09·11에서도 같은 패턴이 반복됩니다.

> ⚠️ 흔한 오해: "블록 수를 늘리면 항상 빠르다"가 아닙니다. `blocks_c=1024`는 GPU의 SM(streaming
> multiprocessor) 개수보다 충분히 많은 블록을 흘려보내 **하드웨어를 다 채우기 위한 것**이지,
> 무한정 늘린다고 좋은 것은 아닙니다 — 적정 규모는 5절의 occupancy 스윕에서 실험적으로 찾습니다.

아래 TODO를 채워보세요. 힌트는 위 코드 스니펫 그대로입니다.


In [ ]:
@cuda.jit
def copy_coalesced(src, dst):
    # TODO: i = cuda.grid(1); stride = cuda.gridsize(1)
    #       while i < src.size: dst[i] = src[i]; i += stride
    pass

blocks_c = 1024   # grid-stride라 블록 수는 자유(충분히 크게)
# GPU 하드웨어를 100% 풀가동시키기 위해서
# copy_coalesced[blocks_c, threads](src, dst)
# cp.cuda.Device().synchronize(); cp.testing.assert_array_equal(src, dst); print('coalesced OK')

<details><summary>💡 해답 보기</summary>

```python
@cuda.jit
def copy_coalesced(src, dst):
    i = cuda.grid(1)
    stride = cuda.gridsize(1)
    while i < src.size:
        dst[i] = src[i]
        i += stride

copy_coalesced[blocks_c, threads](src, dst)
cp.cuda.Device().synchronize()
cp.testing.assert_array_equal(src, dst); print('coalesced OK')
```
</details>

포인트: `stride`가 **모든 스레드에 동일한 값**(`gridDim*blockDim`)이라는 점이 핵심입니다. 매
반복마다 전체 스레드 집합이 **같은 보폭으로 함께 전진**하므로, 어느 반복에서든 warp 내부의
상대적 주소 배치(연속 32개)가 그대로 유지됩니다. 만약 스레드마다 다른 stride를 쓰거나
`i += 1`(2절의 blocked 방식과 사실상 동일)로 바꾸면 이 성질이 깨집니다. `blocks_c=1024`는
데이터 크기(`N=1<<24`)와 무관하게 고정한 값으로, "충분히 많은 블록으로 SM을 가득 채운다"는
목적만 만족하면 되고 나머지는 grid-stride 루프가 알아서 처리합니다.


<a id="4"></a>
## 4. 성능 비교

두 커널을 `bench`로 비교합니다. 같은 복사인데 **접근 패턴**만으로 차이가 납니다(연속 접근이 보통 더 빠름).

### 조금 더 구체적으로

이 벤치마크는 00에서 정의한 `bench`(CUDA 이벤트 기반, 워밍업 포함)를 그대로 재사용합니다 —
GPU 타이밍은 항상 이렇게 재야 한다는 규칙(00 6절)이 커널 작성 단계에서도 동일하게 적용됩니다.
계산이 없는 순수 복사이므로, 두 커널의 성능 차이는 곧 **coalesced 접근이 blocked 접근보다
전역 메모리 트랜잭션을 얼마나 적게 쓰는지**를 그대로 반영합니다.

- **coalesced 복사**는 이론상 GPU의 **최대 메모리 대역폭에 근접**합니다(00에서 언급한 대략
  500 GB/s~2 TB/s 대역, 실제 값은 GPU 세대·모델에 따라 다름) — 읽기 1회 + 쓰기 1회이므로,
  달성 대역폭 ≈ `2 * N * 4바이트 / 실행시간`으로 추정할 수 있습니다.
- **blocked 복사**는 같은 하드웨어에서도 트랜잭션이 여러 배로 쪼개져 유효 대역폭이 그만큼
  낮게 나옵니다. `ipt`(items per thread)가 클수록 한 warp-요청이 커버하는 주소 범위가 넓어져
  격차가 더 벌어지는 경향이 있습니다 — 이 경향은 5·7절의 `items_per_thread` 스윕 실험에서
  직접 확인합니다.
- 실무에서는 이런 손실을 **Nsight Compute의 Memory Workload 리포트**(6절, 선택)에서
  `dram__throughput`·`gld_efficiency` 류의 지표로 정량 확인합니다. 지금 단계에서는 벽시계
  시간(wall/GPU time) 비교만으로도 경향을 충분히 체감할 수 있습니다.

> 실측치는 GPU 모델·드라이버·메모리 상태에 따라 달라지지만, 방향성(coalesced > blocked)은
> 거의 항상 일관됩니다. 만약 차이가 미미하다면 `N`이 작아 커널 런치 오버헤드가 지배적일
> 가능성을 의심해보세요(00의 "일회성 오버헤드", 01의 "손익분기점" 논의와 같은 맥락입니다).


In [ ]:
def run_blocked():   copy_blocked[blocks_b, threads](src, dst, ipt)
def run_coalesced(): copy_coalesced[blocks_c, threads](src, dst)
print_bench(bench(run_blocked,   n_repeat=20, n_warmup=5, name='blocked'))
print_bench(bench(run_coalesced, n_repeat=20, n_warmup=5, name='coalesced'))
print('speedup:', round(gpu_ms(bench(run_blocked))/gpu_ms(bench(run_coalesced)), 2))

### 그림으로 보는 접근 패턴 — 표로 정리하면

앞선 표·비유가 다소 추상적이었다면, 아래는 실제 반복(iteration) 순서대로 어떤 스레드가 어떤
메모리 주소를 만지는지 그대로 나열한 것입니다. 왼쪽 첫 번째 표는 3절의 `copy_coalesced`(grid-stride,
`i += stride`), 두 번째 표는 2절의 `copy_blocked`와 등가인 "연속 구간 배정"(`i += 1`류) 패턴입니다.
숫자를 하나씩 따라가 보면, "같은 열(같은 반복)에서 네 스레드가 어떤 주소를 보는가"가 곧 warp가
그 순간 어디를 접근하는지와 같다는 것을 알 수 있습니다.

- Grid-Stride Loop / Coalesced 코드의 i += stride 방식
```
인접한 스레드(T0, T1, T2, T3)가 같은 시점에 인접한 메모리 주소(0, 1, 2, 3)를 동시에 접근합니다.
스레드 (Thread)  ,1번째 반복 (루프 1),2번째 반복 (루프 2),3번째 반복 (루프 3)
T0             ,메모리 [0] 접근    ,메모리 [4] 접근   ,메모리 [8] 접근
T1             ,메모리 [1] 접근    ,메모리 [5] 접근   ,메모리 [9] 접근
T2             ,메모리 [2] 접근    ,메모리 [6] 접근   ,메모리 [10] 접근
T3             ,메모리 [3] 접근    ,메모리 [7] 접근   ,메모리 [11] 접근
```

- 비연속적 접근 / Uncoalesced
```
만약 코드를 i += 1처럼 작성해서 각 스레드가 자신만의 구역을 연속해서 처리하게 만들면 어떻게 될까요? CPU에서는 이 방식이 일반적이지만, GPU에서는 좋지 않습니다. 
스레드 (Thread),1번째 반복 (루프 1),2번째 반복 (루프 2),3번째 반복 (루프 3)
T0           ,메모리 [0] 접근    ,메모리 [1] 접근   ,메모리 [2] 접근
T1           ,메모리 [3] 접근    ,메모리 [4] 접근   ,메모리 [5] 접근
T2           ,메모리 [6] 접근    ,메모리 [7] 접근   ,메모리 [8] 접근
T3           ,메모리 [9] 접근    ,메모리 [10] 접근  ,메모리 [11] 접근
```

두 표를 **가로(같은 반복)** 로 읽어보세요. 첫 번째 표는 매 반복마다 4개 스레드가 `0,1,2,3` →
`4,5,6,7` → `8,9,10,11`처럼 **항상 연속된 블록**을 함께 요청합니다 — 하드웨어가 이를 하나의
트랜잭션으로 합칠 수 있습니다. 두 번째 표는 매 반복마다 `0,3,6,9`처럼 **3칸씩 떨어진 주소**를
동시에 요청합니다 — 이 4개(실제로는 warp 32개)의 요청은 하나로 합쳐지지 않고 각자 별도
트랜잭션이 필요합니다. 스레드 수가 4개가 아니라 실제 warp 크기인 32개, 그리고 반복 간격(위
예시의 "3")이 실제 코드의 `ipt`(2절 예제에서는 8)로 커지면, 2절에서 계산했던 "32바이트 간격,
1KB 범위"라는 흩어짐이 정확히 이 두 번째 표의 패턴입니다.


<a id="5"></a>
## 5. 파라미터 스윕 (occupancy)

`threads_per_block`을 바꾸며 측정합니다. occupancy·메모리 효율이 달라져 최적점이 생깁니다(07의 7절).

### 조금 더 구체적으로

**occupancy**는 07(7절)에서 정의한 대로 "SM(streaming multiprocessor)에 동시에 상주하는 warp
비율"입니다. `threads_per_block`을 바꾸면 이 비율이 달라지는데, 대략적인 메커니즘은 다음과
같습니다.

- **블록이 너무 작으면(예: 64)**: 블록당 warp 수가 적어(64/32=2 warp), 한 warp가 전역 메모리
  요청을 보내고 응답을 기다리는 동안 스케줄러가 전환할 다른 warp가 부족할 수 있습니다 →
  07(3.1절)의 "CASE 1: Warp 부족"과 같은 상황 — latency hiding이 제대로 안 되어 SM이 놀 수
  있습니다.
- **블록이 너무 크면(예: 1024, 하드웨어 상한)**: 한 SM에 올라갈 수 있는 블록 개수 자체가
  줄어들 수 있고(SM당 최대 상주 스레드 수·레지스터·공유메모리 총량은 제한되어 있음), 이 커널은
  레지스터·공유메모리를 거의 안 쓰므로 크게 문제되지 않을 수 있지만, 복잡한 커널(09·11)에서는
  블록을 키울수록 오히려 SM당 동시 블록 수가 줄어 occupancy가 **낮아지는** 역설이 생깁니다.
- 그래서 흔히 **64~512 threads/block 부근에서 성능 곡선이 완만한 최적 구간**을 이루고, 그
  바깥(너무 작거나 너무 큼)에서 성능이 떨어지는 U자形 또는 완만한 곡선이 관찰되는 경우가
  많습니다. 이 노트북의 복사 커널은 계산이 없고 메모리 접근만 있어 occupancy 민감도가 상대적으로
  낮을 수 있지만, 09(atomic·공유메모리 사용)·11(타일링)처럼 자원을 더 쓰는 커널로 갈수록
  이 튜닝의 중요성이 커집니다.

> 💡 실전 팁: "항상 옳은 정답인 threads_per_block 값"은 없습니다. GPU 세대·커널의 레지스터/공유
> 메모리 사용량·데이터 크기에 따라 최적점이 달라지므로, **직접 스윕해서 확인하는 것**(지금 하는
> 실험처럼)이 표준적인 튜닝 방법입니다. Nsight Compute(6절)의 occupancy 리포트가 이 판단을
> 정량적으로 도와줍니다.


In [ ]:
for threads in [64, 128, 256, 512, 1024]:
    bl = 2048
    def run(t=threads, b=bl): copy_coalesced[b, t](src, dst)
    print_bench(bench(run, n_repeat=20, n_warmup=5, name=f'threads={threads}'))

<a id="6"></a>
## 6. (선택) Nsight Compute 프로파일

<details><summary>펼쳐 보기 — ncu로 메모리 처리량 직접 보기 (GTC 06 워크플로)</summary>

프로파일링은 별도 프로세스 실행이 필요해, 스크립트로 저장 후 커맨드라인 `ncu`로 측정합니다.
강의장에 **Nsight Compute(`ncu`)** 가 있을 때만 동작합니다.

```python
# 주피터 셀 맨 위:  %%writefile copy_bench.py
from numba import cuda; import cupy as cp, numpy as np
@cuda.jit
def copy_coalesced(src,dst):
    i=cuda.grid(1); s=cuda.gridsize(1)
    while i<src.size: dst[i]=src[i]; i+=s
N=1<<26; src=cp.arange(N,dtype=cp.float32); dst=cp.empty_like(src)
copy_coalesced[2048,256](src,dst); cp.cuda.Device().synchronize()
```
```bash
ncu -f --kernel-name regex:copy_coalesced --set full -o copy python copy_bench.py
ncu --import copy.ncu-rep --csv   # 또는 nsightful로 노트북에 표시
```
리포트의 **Memory Workload** 에서 blocked 대비 coalesced의 처리량(throughput)이 높은 것을 확인하세요.
</details>

**왜 `%%writefile`로 별도 스크립트를 만드는가**: `ncu`는 대상 프로세스를 **처음부터 자신이
직접 실행**하며 매 CUDA API 호출을 가로채 계측해야 하므로, 이미 떠 있는 주피터 커널 프로세스에
"붙어서" 재는 것이 아니라 **새 파이썬 프로세스를 통째로 실행**시켜야 합니다. 그래서 코드를
파일로 저장한 뒤 `ncu ... python copy_bench.py`처럼 커맨드라인에서 그 스크립트를 실행하는
간접 방식을 씁니다 — 05·06에서 다룬 `cupyx.profiler.time_range`/NVTX가 "실행 중인 프로세스
안에서" 구간을 표시하는 것과는 결이 다른 접근입니다.

`--set full` 리포트의 **Memory Workload Analysis** 섹션에서 특히 눈여겨볼 지표:
- **Memory Throughput**: 이 커널이 실제로 달성한 대역폭(GB/s) — coalesced가 blocked보다
  뚜렷하게 높게 나와야 합니다.
- **L2/DRAM Hit Rate**, **Sectors per Request**: 요청당 몇 개의 32바이트 섹터를 실제로 썼는지 —
  blocked 커널은 이 값이 비효율적으로(필요한 것보다 많이) 높게 나옵니다.
- 이 지표들이 4절에서 관찰한 벽시계 시간 차이의 **원인을 직접 숫자로 확인**시켜 줍니다.


## 🧪 추가 연습 & 실험

아래는 본문에서 다룬 blocked/coalesced 복사를 다른 각도(occupancy·다른 커널·다른 접근 패턴)로
변주해보는 선택 실습입니다. 시간이 허락하는 만큼 골라서 풀어보세요 — 모두 07~08에서 세운
coalescing·occupancy 개념을 새 상황에 적용해보는 연습입니다.


**실험 — items_per_thread 스윕**: blocked copy에서 스레드당 처리량을 바꿔 측정해 보세요(접근 패턴·occupancy 영향).

> 예측 먼저: `ipt`(스레드당 처리 원소 수)가 커지면 2절에서 계산한 "warp가 흩어져 보는 주소
> 범위"가 `ipt`에 비례해 넓어집니다(예: `ipt=8`이면 1KB 범위, `ipt=32`면 4KB 범위). 트랜잭션
> 효율이 더 나빠질까요, 아니면 스레드 수가 줄어 런치 오버헤드가 줄면서 상쇄될까요? 직접 돌려서
> 확인하세요.


In [ ]:
for ipt in [1, 4, 8, 16, 32]:
    bb = (N + threads*ipt - 1)//(threads*ipt)
    def run(b=bb, k=ipt): copy_blocked[b, threads](src, dst, k)
    print('ipt', ipt, '->', round(gpu_ms(bench(run, n_repeat=20, n_warmup=5)), 4), 'ms')

**연습 — saxpy Numba 커널**: `y = a*x + b` 를 grid-stride로 작성하세요(coalesced).

**saxpy**(Single-precision A·X Plus Y)는 BLAS Level-1의 대표 연산으로, 00(3절)에서 언급한
cuBLAS가 내부적으로 최적화해 제공하는 바로 그 연산군에 속합니다. 평소라면 `a*x+b`를 CuPy
표현식이나 `cp.ElementwiseKernel`(07의 9절)로 한 줄에 처리하겠지만, 여기서는 **grid-stride
coalesced 패턴이 복사 이외의 커널에도 그대로 적용된다는 것**을 직접 손으로 확인하는 것이 목적입니다.
3절의 `copy_coalesced`와 구조가 거의 동일하되, `dst[i]=src[i]` 대신 `y[i]=a*x[i]+b`라는
계산이 추가될 뿐입니다.


In [ ]:
@cuda.jit
def saxpy(x, y, a, b):
    # TODO: i=cuda.grid(1); s=cuda.gridsize(1); while i<x.size: y[i]=a*x[i]+b; i+=s
    pass
xx = cp.random.rand(1<<22, dtype=cp.float32); yy = cp.empty_like(xx)
# saxpy[1024,256](xx,yy,np.float32(2),np.float32(1)); cp.cuda.Device().synchronize()
# cp.testing.assert_allclose(cp.asnumpy(yy), 2*cp.asnumpy(xx)+1, rtol=1e-5); print('OK')

<details><summary>💡 해답 보기</summary>

```python
@cuda.jit
def saxpy(x, y, a, b):
    i = cuda.grid(1); s = cuda.gridsize(1)
    while i < x.size:
        y[i] = a*x[i] + b; i += s
saxpy[1024,256](xx, yy, np.float32(2), np.float32(1)); cp.cuda.Device().synchronize()
cp.testing.assert_allclose(cp.asnumpy(yy), 2*cp.asnumpy(xx)+1, rtol=1e-5); print('OK')
```
</details>

포인트: 3절의 `copy_coalesced`와 몸통이 사실상 동일합니다(`i`, `stride` 계산·`while` 루프
구조가 그대로). 바뀐 것은 대입문 오른쪽의 **계산식뿐**입니다. 이는 grid-stride 루프가 "복사
전용 트릭"이 아니라, **원소별(element-wise) 연산 전반에 적용되는 범용 coalescing 패턴**임을
보여줍니다 — 09(히스토그램)에서는 여기에 atomic 연산이, 11(RawKernel)에서는 공유메모리 타일링이
추가되지만, "인접 스레드가 인접 주소를 보게 만든다"는 이 핵심 아이디어는 계속 재사용됩니다.


**연습 — 2D naive transpose**: 행을 읽어 열에 쓰는 전치 커널. 쓰기가 **비연속(non-coalesced)** 임을 관찰하세요(11에서 타일로 개선).

전치(transpose)는 이 노트북의 1D 복사와 달리 **2차원 인덱싱**(`cuda.grid(2)`)을 쓰는 첫 예제이며,
동시에 "읽기는 coalesced인데 쓰기는 non-coalesced"인, **하나의 커널 안에 두 가지 접근 패턴이
공존하는** 흥미로운 사례입니다. `out[j,i] = a[i,j]`에서 `a[i,j]`를 읽을 때는 `i`(행 내부 위치)가
연속으로 증가하며 warp가 인접 주소를 읽지만(coalesced), 같은 warp가 `out[j,i]`에 쓸 때는
`j`가 고정된 채 `i`가 바뀌므로 결과적으로 **행렬의 열 방향으로, 즉 `n`칸씩 떨어진 주소에** 씁니다
(non-coalesced). 읽기·쓰기 양쪽을 모두 coalesced로 만들려면 **공유메모리에 타일 단위로 한 번
모아뒀다가(coalesced 읽기) 전치된 순서로 공유메모리에서 다시 꺼내 쓰는(coalesced 쓰기)** 기법이
필요한데, 이것이 바로 `11_rawkernel`에서 다루는 **타일 전치(tiled transpose)** 최적화입니다.
지금은 "문제가 어디서 생기는지"를 눈으로 확인하는 것이 목표입니다.


In [ ]:
@cuda.jit
def transpose_naive(a, out, n):
    # TODO: i,j = cuda.grid(2); if i<n and j<n: out[j,i] = a[i,j]
    pass
n=2048; A=cp.random.rand(n,n,dtype=cp.float32); T=cp.empty_like(A)
tpb=(16,16); bpg=(math.ceil(n/16), math.ceil(n/16))
# transpose_naive[bpg,tpb](A,T,np.int32(n)); cp.cuda.Device().synchronize()
# cp.testing.assert_array_equal(cp.asnumpy(T), cp.asnumpy(A).T); print('OK')

<details><summary>💡 해답 보기</summary>

```python
@cuda.jit
def transpose_naive(a, out, n):
    i, j = cuda.grid(2)
    if i < n and j < n:
        out[j, i] = a[i, j]    # 읽기는 연속, 쓰기는 비연속
transpose_naive[bpg,tpb](A,T,np.int32(n)); cp.cuda.Device().synchronize()
cp.testing.assert_array_equal(cp.asnumpy(T), cp.asnumpy(A).T); print('OK')
```
</details>

포인트: `cuda.grid(2)`가 반환하는 `(i, j)`는 관례상 `(x, y)`에 대응하며, 여기서는 `i`를 행,
`j`를 열로 사용했습니다. warp는 보통 x 방향(`i`, `threadIdx.x`)으로 연속된 스레드들로
구성되므로, `a[i,j]`처럼 **첫 번째 첨자가 warp 방향과 같은 축이면** 그 접근은 coalesced가
됩니다. 반대로 `out[j,i]`처럼 warp 방향 인덱스(`i`)가 **행렬의 열(두 번째 첨자)** 로 들어가면,
같은 warp의 스레드들은 서로 `n`개(한 행 전체 길이)만큼 떨어진 주소에 쓰게 되어 non-coalesced가
됩니다. `n=2048`이면 이 간격은 `2048*4바이트=8KB`로, 2절의 blocked 복사(간격 32바이트)보다
훨씬 극단적인 흩어짐입니다 — 그만큼 성능 손실도 클 수 있습니다.


<a id="7"></a>
## 7. 체크포인트

이 노트북에서는 커널을 "CuPy가 대신 만들어주는" 07의 세계에서 벗어나, **스레드가 정확히 무엇을
하는지 한 줄 한 줄 직접 통제**하는 세계로 넘어왔습니다. 그 첫 결실이 blocked vs coalesced의
성능 격차이고, occupancy 스윕은 "런치 구성도 최적화 대상"이라는 감각을 더해줍니다.

- [ ] `@cuda.jit`+`cuda.grid`로 커널을 작성·런치했다
- [ ] blocked vs coalesced **메모리 접근 패턴**의 성능 차이를 측정했다
- [ ] grid-stride 루프로 coalesced 복사를 구현했다
- [ ] threads_per_block 스윕으로 occupancy 영향을 봤다
- [ ] (선택) ncu로 메모리 처리량을 확인했다

다음: **`09_numba_histogram`** — atomic·공유메모리·동기화로 히스토그램을 단계적으로 최적화합니다.
